In [0]:
table_df = spark.table(
    "accelerator.metadata.table_configs"
)
table_df.display()

In [0]:
schema_data = []

for row in table_df.collect():

    table_name = row["table_name"]
    table_name = f"accelerator.gold.bronze_{table_name}"
    # print(table_name)
    df = spark.table(table_name)
    # print(df)
    # print(df.schema)
    # print(df.schema.fields)
    for field in df.schema.fields:

        schema_data.append(
            (
                str(table_name).replace('accelerator.gold.bronze_', ''),
                field.name,
                str(field.dataType)
            )
        )
    print(f"Schema Data: {schema_data}")

In [0]:
from pyspark.sql.functions import current_timestamp
# creating snapshot of this
current_schema_df = spark.createDataFrame(
    schema_data,
    [
        "table_name",
        "column_name",
        "data_type"
    ]
).withColumn(
    "captured_date",
    current_timestamp()
)

In [0]:
current_schema_df.head(20)

In [0]:
# current_schema_df.write \
#     .mode("append") \
#     .saveAsTable(
#         "accelerator.metadata.schema_history"
#     )

In [0]:
spark.read.table('accelerator.metadata.schema_history').display()

In [0]:
history_df = spark.table(
    "accelerator.metadata.schema_history"
)
from pyspark.sql.functions import max

latest_date = history_df.select(
    max("captured_date")
).collect()[0][0]


print(latest_date)

previous_schema_df = history_df.filter(
    history_df.captured_date == latest_date
)

previous_schema_df.head(20)

In [0]:
previous_schema_df.display()
current_schema_df.display()


In [0]:
new_columns = (
    current_schema_df
    .select(
        "table_name",
        "column_name",
        "data_type"
    )
    .subtract(
        previous_schema_df.select(
            "table_name",
            "column_name",
            "data_type"
        )
    )
)

In [0]:
display(new_columns)

In [0]:
from pyspark.sql.functions import lit,current_timestamp

drift_df = (
    new_columns
    .withColumn(
        "drift_type",
        lit("NEW_COLUMN")
    )
    .withColumn(
        "old_data_type",
        lit(None)
    )
    .withColumnRenamed(
        "data_type",
        "new_data_type"
    )
    .withColumn(
        "detected_on",
        current_timestamp()
    )
)

In [0]:
drift_df.display()

In [0]:
drift_df.select(
    "table_name",
    "drift_type",
    "column_name",
    "old_data_type",
    "new_data_type",
    "detected_on"
).write.mode("append").saveAsTable(
    "accelerator.metadata.schema_drift_audit"
)

In [0]:
from pyspark.sql.functions import col
spark.read.table("accelerator.metadata.schema_drift_audit").withColumn("old_data_type", col("old_data_type").cast("string")).display()

In [0]:
%sql
alter table accelerator.metadata.schema_drift_audit
alter column old_data_type [string];

In [0]:
current_schema_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "accelerator.metadata.schema_history"
    )